# 09 · Compile results

Pulls every `results/*.json` from notebooks 04–08 into one master file,
`results/analysis_results.json`, plus a sample-size / statistical power
calculation. This is the single source of truth for every number quoted
in the README and the Power BI dashboard.

In [1]:
import json
from pathlib import Path
from scipy import stats

ROOT = Path("..").resolve()
RESULTS = ROOT / "results"


def load(name):
    with open(RESULTS / name) as f:
        return json.load(f)


rfm = load("rfm_summary.json")
cohort = load("cohort_summary.json")
delivery = load("delivery_summary.json")
hyp = load("hypothesis_test_results.json")
seller_cat = load("seller_category_summary.json")

## Sample-size / power calculation
Answers: "how many orders would we need to sample to detect a 1-day
improvement in delivery time at 95% confidence?" — using this dataset's
actual std dev of delivery time.

In [2]:
sigma = delivery["national"]["std_delivery_days"]
delta = 1.0
alpha = 0.05
power = 0.80
z_a1 = stats.norm.ppf(1 - alpha)
z_a2 = stats.norm.ppf(1 - alpha / 2)
z_b = stats.norm.ppf(power)
power_calc = {
    "input_std_delivery_days": sigma,
    "detectable_improvement_days": delta,
    "alpha": alpha,
    "power": power,
    "n_one_sample_one_sided": round(((z_a1 + z_b) * sigma / delta) ** 2),
    "n_one_sample_two_sided": round(((z_a2 + z_b) * sigma / delta) ** 2),
    "n_per_group_two_sample_two_sided": round(2 * ((z_a2 + z_b) * sigma / delta) ** 2),
}
print(json.dumps(power_calc, indent=2))

{
  "input_std_delivery_days": 9.55,
  "detectable_improvement_days": 1.0,
  "alpha": 0.05,
  "power": 0.8,
  "n_one_sample_one_sided": 564,
  "n_one_sample_two_sided": 716,
  "n_per_group_two_sample_two_sided": 1432
}


In [3]:
master = {
    "dataset": {
        "source": "Olist Brazilian E-Commerce Public Dataset (2016-09 to 2018-10)",
        "n_orders": 99441,
        "n_delivered_orders": delivery["national"]["n_delivered_orders"],
        "n_customers_unique": rfm["n_customers"],
        "n_sellers": seller_cat["n_total_sellers"],
        "total_source_rows_loaded": 1551698,
    },
    "rfm_segmentation": rfm,
    "cohort_retention": cohort,
    "delivery_performance": delivery,
    "hypothesis_testing": hyp,
    "seller_category": seller_cat,
    "sample_size_power_calc": power_calc,
}

with open(RESULTS / "analysis_results.json", "w") as f:
    json.dump(master, f, indent=2, default=str)

print("Wrote results/analysis_results.json")

Wrote results/analysis_results.json
